# Lab 10 · Đảm bảo chất lượng chéo bảng

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 10**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 10 làm sạch bảng `listings`. Lab này kiểm tra tính nhất quán giữa
hai bảng thông qua khoá ngoại, khoá tự nhiên và các cột dẫn xuất có sẵn.

*Từ tuần này phần kỹ năng kéo dài khoảng 50 phút; 30 phút cuối dành cho hoạt động hỗ trợ bài tập lớn.*

## Cách làm việc trong bài lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau bài lab, bạn sẽ:

1. Kiểm tra miền thời gian và khoá ngoại giữa hai bảng, rồi diễn giải đúng kết quả của phép kiểm.
2. Chứng minh một bảng **không có khoá tự nhiên** và nói được hệ quả.
3. Đối chiếu cột dẫn xuất có sẵn với con số tự đếm và phân tích phần chênh lệch.
4. Đóng gói mọi phép kiểm thành `qa_report` ghi ra file.

In [ ]:
import pandas as pd

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
ds = pd.read_csv(f"{BASE}/listings.csv", parse_dates=["last_review"])
rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
len(ds), len(rv)

### Bước 1 · Miền thời gian của bảng đánh giá

Tên thư mục ghi mốc chụp **29/06/2026**. Các đánh giá có ngày sau mốc đó cần được gắn cờ.

In [ ]:
# TODO: tìm ngày đánh giá sớm nhất, muộn nhất; đếm dòng CÓ NGÀY SAU 2026-06-29
ngay_dau = ...
ngay_cuoi = ...
so_sau_moc = ...

# --- Ô kiểm tra ---
assert str(ngay_dau)[:10] == "2010-11-13"
assert str(ngay_cuoi)[:10] == "2026-07-01"
assert so_sau_moc == 204
print(f"Ngày đánh giá từ {ngay_dau:%Y-%m-%d} đến {ngay_cuoi:%Y-%m-%d}; {so_sau_moc} dòng sau mốc chụp.")

Có **204 dòng sau mốc chụp**: 155 dòng ngày 30/06 và 49 dòng ngày 01/07. Đây là độ lệch
giữa nhãn mốc chụp và miền ngày trong tệp. `rv["date"].max()` là một phép chẩn đoán,
không tự động thay thế mốc 29/06 bằng 01/07. Với phân tích thời gian ở Bài 8, ta chọn
mốc công bố 29/06 để các kỳ so sánh nhất quán và loại 204 dòng này. Quy trình của nhóm
phải nêu rõ chính sách chọn mốc và áp dụng cùng một cách cho mọi bảng.

### Bước 2 · Khoá ngoại: đánh giá không khớp chỗ ở

Mọi `listing_id` trong bảng đánh giá phải tồn tại trong `ds["id"]`. Nếu không,
hai bảng không bảo toàn khoá ngoại.

In [ ]:
# TODO: đếm dòng có listing_id KHÔNG nằm trong ds["id"] (dùng isin và ~)
so_mo_coi = ...

# --- Ô kiểm tra ---
assert so_mo_coi == 0
print("0 đánh giá không khớp — khoá ngoại toàn vẹn.")

Phép kiểm này có **0 vi phạm** và vẫn phải được ghi trong báo cáo. Số 0 cho biết quy tắc
đã chạy và không phát hiện vấn đề, khác với việc chưa chạy phép kiểm.

### Bước 3 · Khoá tự nhiên: bảng đánh giá rút gọn có khoá không?

In [ ]:
# TODO: đếm số dòng trùng theo cặp (listing_id, date) — duplicated với subset
so_trung_cap = ...

# --- Ô kiểm tra ---
assert so_trung_cap == 3548
print(f"{so_trung_cap:,} dòng 'trùng' theo (listing_id, date).")

3.548 dòng có cùng `(listing_id, date)`, nhưng hai khách khác nhau có thể đánh giá cùng
một chỗ ở trong cùng ngày. Bảng rút gọn này **không có khoá tự nhiên**; bản đầy đủ có
cột `id` cho từng đánh giá. Không được dùng `drop_duplicates` theo hai cột trên vì sẽ
xoá các đánh giá hợp lệ.

### Bước 4 · Đối chiếu cột dẫn xuất với con số tự đếm

Bảng `listings` có sẵn `number_of_reviews` và `number_of_reviews_ltm` (LTM, 12 tháng gần nhất)
do Inside Airbnb tính. Ta sẽ tự đếm từ bảng đánh giá để kiểm tra định nghĩa và độ nhất quán.

In [ ]:
# TODO: đếm số đánh giá của từng chỗ ở từ bảng rv (groupby listing_id + size),
#       đưa về ds theo id (map hoặc reindex), điền 0 cho chỗ ở không có đánh giá
dem_that = ...
ds["dem_that"] = ...

# TODO: đếm số chỗ ở có number_of_reviews KHÁC dem_that
so_lech_tong = ...

# --- Ô kiểm tra ---
assert so_lech_tong == 0
print("18.534/18.534 chỗ ở: number_of_reviews khớp số đếm từ toàn bộ bảng rv.")

Việc khớp trên toàn bộ bảng cho thấy `number_of_reviews` cũng bao gồm 204 dòng sau mốc 29/06.
Đây là tính nhất quán nội bộ giữa hai tệp, không phải bằng chứng rằng mốc công bố phải đổi
thành 01/07. Nếu quy trình chọn cắt tại 29/06, các cột dẫn xuất cũng phải được tính lại từ
bảng đánh giá đã cắt theo cùng chính sách.

In [ ]:
# Với LTM (last twelve months), thử định nghĩa khoảng 12 tháng từ 2025-06-30:
ltm_that = rv[rv["date"] >= "2025-06-30"].groupby("listing_id").size()
ds["ltm_that"] = ds["id"].map(ltm_that).fillna(0).astype(int)

# TODO: đếm số chỗ ở lệch giữa number_of_reviews_ltm và ltm_that,
#       rồi xem phân bố (cột có sẵn − số tự đếm) bằng describe
so_lech_ltm = ...
do_lech = ...

# --- Ô kiểm tra ---
assert so_lech_ltm == 665
print(f"{so_lech_ltm} chỗ ở lệch LTM; độ lệch trung vị toàn bảng: {do_lech.median():.0f}")
do_lech[do_lech != 0].describe().round(2)

Có 665 chỗ ở lệch; trong nhóm lệch, phần lớn có độ lệch **−1**. Kết quả này phù hợp với
khả năng Inside Airbnb dùng một mốc hoặc quy ước cửa sổ 12 tháng khác, nhưng chưa đủ để
khẳng định nguyên nhân. Khi dùng cột dẫn xuất, cần tra định nghĩa gốc hoặc tự tính theo
một định nghĩa đã ghi rõ. Phân bố độ lệch giúp phân biệt sai khác có cấu trúc với sai khác
không ổn định.

### Bước 5 · Đóng gói qa_report

In [ ]:
# TODO: gom 5 phép kiểm thành qa_report (DataFrame 3 cột: quy_tac, so_dong, ghi_chu)
#       rồi ghi ra file qa_report_reviews.csv
qa_report = pd.DataFrame([
    {"quy_tac": "review_sau_moc",      "so_dong": so_sau_moc,    "ghi_chu": "date > mốc công bố; áp dụng chính sách cắt mốc đã ghi rõ"},
    {"quy_tac": "review_mo_coi",      "so_dong": so_mo_coi,     "ghi_chu": "listing_id không có trong listings"},
    {"quy_tac": "trung_theo_cap",     "so_dong": so_trung_cap,  "ghi_chu": "không xoá; hai cột không phải khoá tự nhiên"},
    {"quy_tac": "lech_tong_review",   "so_dong": so_lech_tong,  "ghi_chu": "cột dẫn xuất so với số tự đếm"},
    {"quy_tac": "lech_ltm",           "so_dong": so_lech_ltm,   "ghi_chu": "khác định nghĩa mốc 12 tháng"},
])
# TODO: ghi ra CSV (không index) và đọc lại để kiểm
...

# --- Ô kiểm tra ---
kq = pd.read_csv("qa_report_reviews.csv")
assert kq.shape == (5, 3)
assert kq.set_index("quy_tac")["so_dong"].to_dict() == {
    "review_sau_moc": 204, "review_mo_coi": 0, "trung_theo_cap": 3548,
    "lech_tong_review": 0, "lech_ltm": 665,
}
qa_report

## Bài tự làm ✅ mở

**Bộ QA trên mốc chụp khác.** Chạy lại năm phép kiểm trên mốc **2025-09-27**
(đổi `BASE` thành `.../2025-09-27/visualisations`; cập nhật mốc cắt và cửa sổ LTM).
Quy tắc nào cho kết quả khác? Ghi lại thay đổi cần thiết để cùng một bộ quy tắc chạy được
trên một mốc chụp khác (bộ test chung khi chấm có thể dùng mốc như vậy).

In [ ]:
# Viết bài tự làm của bạn ở đây

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm việc theo nhóm)

Tự soát theo checklist; đánh dấu ✅/❌ và ghi một dòng "việc tuần tới" cho mỗi ❌:

1. ☐ Repo GitHub **riêng tư** đã mời tài khoản giảng viên và giảng viên thực hành
   (danh sách trên Canvas Portal); cấu trúc thư mục đúng theo đề.
2. ☐ **Mã tải dữ liệu** chạy được: tải đủ các mốc chụp bắt buộc của thành phố nhóm
   vào `data/raw/` — chưa commit dữ liệu thô lên repo.
3. ☐ **Kế hoạch phân tích** (bên đặt hàng, câu hỏi, KPI dự kiến, kế hoạch LLM) đã viết ra,
   cả nhóm thống nhất và chia thành đầu việc có người phụ trách.
4. ☐ **Bộ quy tắc QA nháp** (theo bài 10 + lab này): mỗi quy tắc đủ 4 phần
   tên–điều kiện–lý do–hành động; chạy thử trên mốc chụp mới nhất và tạo `qa_report` đầu tiên.
5. ☐ Phân công tuần tới: ai phụ trách **hợp phần LLM** (bài 11) — người đó tạo API key
   Gemini trước ở nhà (aistudio.google.com, miễn phí).

> Nhóm hoàn thành sớm có thể chạy bộ QA trên **một mốc chụp chưa dùng** để kiểm tra
> khả năng áp dụng trên mốc chụp lạ.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Kiểm miền thời gian và khoá ngoại | `qa_report` của bài tập lớn |
| Chứng minh bảng không có khoá tự nhiên | tránh dùng `drop_duplicates` máy móc |
| Đối chiếu cột dẫn xuất với số tự đếm | kiểm tra định nghĩa trước khi sử dụng |
| `qa_report.csv` | sản phẩm theo yêu cầu của đề |

Bài giảng tới: **mô hình ngôn ngữ lớn (LLM) cho dữ liệu phi cấu trúc**.
Hãy chuẩn bị API key Gemini theo hướng dẫn ở đầu notebook Bài 11.